# Prostate MRI 4-Model Stacking Ensemble

This notebook runs on the **AIIMS prostate MRI prepared dataset**.

**Dataset target supported by the supplied README:** binary lesion classification:
- `0` = PI-RADS 1–2
- `1` = PI-RADS 3–5

The MRI inputs are:
- T2: `t2.nii.gz`
- ADC: `adc.nii.gz`
- high-b DWI: `dwi_highb.nii.gz`

All three are on the same T2 voxel grid.

### Ensemble
1. Swin UNETR (MONAI encoder, self-supervised pretrained on BTCV/CT data)
2. MedicalNet 3D ResNet-50 (pretrained on 23 medical imaging datasets)
3. DenseNet-121 (ImageNet pretrained)
4. InceptionResNetV2 (ImageNet pretrained)
5. Logistic-regression stacking meta-learner

The 3D models consume a multimodal 3D tensor `[T2, ADC, DWI]`.
The 2D models use a 2.5D representation built from neighboring T2 axial slices.

### Transfer learning strategy: frozen backbones + augmented linear probes

Each of the four base models' pretrained backbone is **frozen** (no backprop through it, ever) and used only to extract a pooled feature vector per patient (3D models) or per slice (2D models). A small **logistic-regression linear probe** is then fit per model directly on those frozen features -- that's the actual fine-tuning / transfer-learning step.

Two things stretch the small amount of labeled data here as far as it reasonably goes:

- **All labeled patients, not just `train_df`.** `train_df` + `val_df` (67 patients total) are combined and used via **K-fold cross-validation** (CELL 22): each fold fits probes on the other folds only, then predicts the held-out fold, so every labeled patient gets an out-of-fold (OOF) prediction from a probe that never saw it. The stacking meta-learner (#5) is fit on those OOF predictions — using every labeled patient, with no leakage. "Production" probes used for the actual test run are then refit on *all* 67 labeled patients.
- **Data augmentation when fitting probes.** Each patient/slice contributes `1 + N_AUG` feature vectors to probe training -- the original plus `N_AUG` randomly flipped/zoom-cropped/intensity-jittered copies (`augment_volume_3d` / `augment_slice_2d`). The frozen backbone still only ever does a forward pass (no gradients), so this is cheap; augmentation is never applied when generating OOF, validation, or test predictions, so reported metrics stay honest.

Why linear probing instead of full fine-tuning: even with augmentation, this is nowhere near enough data to fine-tune a 3D ResNet-50 or a Swin transformer end-to-end without overfitting -- and it's also the most expensive way to use a 4 GB GPU. A linear probe on frozen, already-good pretrained features is the standard, data-efficient alternative -- and since features are cached to `ensemble_outputs/features/`, re-fitting a probe with different hyperparameters costs milliseconds, not another pass through the network.

> **Still read this before trusting a score.** None of these backbones were pretrained *on prostate cancer classification* -- MedicalNet was pretrained for segmentation across 23 varied medical datasets, the Swin encoder was pretrained self-supervised (no labels at all) on unlabeled CT, and the two 2D models were pretrained on ImageNet photos. "Pretrained" here means each backbone learned generally useful visual features; the part that actually learned to call this dataset's lesions lesion-positive is the small linear probes fit above, on 67 labeled patients (plus augmented copies). That's a real, somewhat stronger training signal than before -- expect "clearly better than random guessing," not "clinically deployable."


In [1]:
import torch

In [2]:
# CELL 2 — Imports and global configuration

import os, sys, json, math, random, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# CHANGE ONLY THIS BLOCK (if your layout differs)
# ============================================================

# Auto-detect the dataset root so the notebook runs unchanged on
# Windows, WSL, or plain Linux. Falls back to the notebook's own
# working directory if none of the known layouts match.
_CANDIDATE_ROOTS = [
    Path(r"C:\Users\adity\Downloads\aiims_public"),
    Path("/mnt/c/Users/adity/Downloads/aiims_public"),
    Path.cwd(),
]
DATA_ROOT = next((p for p in _CANDIDATE_ROOTS if (p / "images").exists()), Path.cwd())

IMAGE_ROOT = DATA_ROOT / "images"
MASK_ROOT  = DATA_ROOT / "masks"

LABELS_CSV = DATA_ROOT / "labels_dev.csv"
SPLIT_CSV  = DATA_ROOT / "split.csv"
SLICE_LABELS_CSV = DATA_ROOT / "slice_labels_dev.csv"

EXTERNAL_MODELS_DIR = DATA_ROOT / "external_models"
sys.path.insert(0, str(EXTERNAL_MODELS_DIR))

CHECKPOINT_DIR = DATA_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MEDICALNET_CKPT = CHECKPOINT_DIR / "medicalnet_resnet50_23datasets.pth"
SWIN_CKPT = CHECKPOINT_DIR / "model_swinvit.pt"

# Public, pretrained (NOT fine-tuned on this dataset) weight sources.
# DenseNet-121 and InceptionResNetV2 use standard ImageNet pretrained
# weights (auto-downloaded by torchvision / timm) instead of RadImageNet,
# since no safe, state-dict-only PyTorch RadImageNet checkpoint for
# InceptionResNetV2 is publicly available.
MEDICALNET_URL = "https://huggingface.co/MONAI/checkpoints/resolve/main/medicalnet_resnet50_23datasets.pth"
SWIN_SSL_URL = "https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/model_swinvit.pt"

OUTPUT_DIR = DATA_ROOT / "ensemble_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def ensure_checkpoint(path: Path, url: str):
    """Downloads a pretrained checkpoint if it isn't already on disk."""
    if path.exists():
        return path
    print(f"Downloading pretrained weights to {path} ...")
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.hub.download_url_to_file(url, str(path))
    return path


# Memory-safe settings for a 4 GB RTX 3050 Ti.
# Each spatial dim must be divisible by 32 for Swin UNETR's 5 downsampling
# stages (2 patch-embed + 4 window-merging halvings).
PATCH_SIZE = (96, 96, 64)      # adjust if OOM, keep each dim a multiple of 32
BATCH_3D = 1
BATCH_2D = 8

NUM_WORKERS = 0                # safest on Windows
PIN_MEMORY = torch.cuda.is_available()

# 2.5D slice settings
NEIGHBOR_RADIUS = 1            # k-1, k, k+1

# Frozen-feature linear-probe training settings (train+val combined via
# K-fold cross-validation, with augmented copies per patient/slice -- see
# the "frozen-feature linear probes" cell for why).
N_FOLDS = 5
N_AUG = 4                      # augmented copies per patient/slice, in addition to the original

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
DATA_ROOT: C:\Users\adity\Downloads\aiims_public
OUTPUT_DIR: C:\Users\adity\Downloads\aiims_public\ensemble_outputs


In [3]:
# CELL 3 — Read the actual CSVs and verify their schema

labels = pd.read_csv(LABELS_CSV)
splits = pd.read_csv(SPLIT_CSV)
slice_labels = pd.read_csv(SLICE_LABELS_CSV)

print("labels_dev:")
display(labels.head())
print(labels.columns.tolist())

print("\nsplit:")
display(splits.head())
print(splits.columns.tolist())

print("\nslice_labels_dev:")
display(slice_labels.head())
print(slice_labels.columns.tolist())

# Detect common column names without silently changing labels.
def find_col(df, candidates, required=True):
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"Could not find any of {candidates} in columns {list(df.columns)}")
    return None

CODE_COL_LABEL = find_col(labels, ["code", "patient", "patient_code", "id"])
TARGET_COL = find_col(labels, ["label", "target", "lesion", "y"])
CODE_COL_SPLIT = find_col(splits, ["code", "patient", "patient_code", "id"])
SPLIT_COL = find_col(splits, ["split", "partition", "set"])

print("\nDetected:")
print("label code:", CODE_COL_LABEL)
print("target:", TARGET_COL)
print("split code:", CODE_COL_SPLIT)
print("split:", SPLIT_COL)

print("\nTarget distribution:")
print(labels[TARGET_COL].value_counts(dropna=False).sort_index())
print("\nSplit distribution:")
print(splits[SPLIT_COL].value_counts(dropna=False))


labels_dev:


,code,split,label,pirads_group
0,C0001,val,1,4
1,C0002,train,0,1-2
2,C0003,val,1,3
3,C0008,val,0,1-2
4,C0009,val,0,1-2


['code', 'split', 'label', 'pirads_group']

split:


,code,split
0,C0001,val
1,C0002,train
2,C0003,val
3,C0004,test
4,C0005,test


['code', 'split']

slice_labels_dev:


,code,slice_type,slice_index,mask_label_value,pirads,size_mm,side
0,C0001,lesion,15,1,4.0,22.6,Lt
1,C0001,lesion,16,1,4.0,22.6,Lt
2,C0001,lesion,17,1,4.0,22.6,Lt
3,C0001,lesion,18,1,4.0,22.6,Lt
4,C0001,normal,15,2,NaN,NaN,NaN


['code', 'slice_type', 'slice_index', 'mask_label_value', 'pirads', 'size_mm', 'side']

Detected:
label code: code
target: label
split code: code
split: split

Target distribution:
label
0    42
1    25
Name: count, dtype: int64

Split distribution:
split
train    38
test     30
val      29
Name: count, dtype: int64


In [4]:
# CELL 4 — Merge patient labels with patient split

labels2 = labels[[CODE_COL_LABEL, TARGET_COL]].copy()
labels2.columns = ["code", "target"]

splits2 = splits[[CODE_COL_SPLIT, SPLIT_COL]].copy()
splits2.columns = ["code", "split"]

patients = splits2.merge(labels2, on="code", how="left")

print("Total patients:", len(patients))
print(patients["split"].value_counts(dropna=False))

print("\nLabels available by split:")
print(pd.crosstab(patients["split"], patients["target"], dropna=False))

# Test labels are expected to be hidden.
print("\nMissing targets:")
print(patients["target"].isna().sum())


Total patients: 97
split
train    38
test     30
val      29
Name: count, dtype: int64

Labels available by split:
target  0.0  1.0  NaN
split                
test      0    0   30
train    24   14    0
val      18   11    0

Missing targets:
30


In [5]:
# CELL 5 — NIfTI loading and robust MRI normalization

def load_nifti(path):
    img = nib.load(str(path))
    arr = img.get_fdata(dtype=np.float32)
    return arr, img.affine

def robust_zscore(x, mask=None, low=1.0, high=99.0, eps=1e-6):
    """
    Percentile clipping followed by z-score normalization.
    MRI intensities are not assumed to be normalized.
    """
    x = np.nan_to_num(x.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

    if mask is None:
        valid = np.isfinite(x) & (x != 0)
    else:
        valid = mask.astype(bool) & np.isfinite(x)

    vals = x[valid]
    if vals.size < 16:
        vals = x[np.isfinite(x)]

    if vals.size == 0:
        return np.zeros_like(x, dtype=np.float32)

    lo, hi = np.percentile(vals, [low, high])
    x = np.clip(x, lo, hi)

    vals2 = x[valid]
    mean = vals2.mean()
    std = vals2.std()

    return ((x - mean) / (std + eps)).astype(np.float32)

def load_patient_volume(code):
    patient_dir = IMAGE_ROOT / str(code)

    t2_path = patient_dir / "t2.nii.gz"
    adc_path = patient_dir / "adc.nii.gz"
    dwi_path = patient_dir / "dwi_highb.nii.gz"

    for p in [t2_path, adc_path, dwi_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing MRI file: {p}")

    t2, _ = load_nifti(t2_path)
    adc, _ = load_nifti(adc_path)
    dwi, _ = load_nifti(dwi_path)

    if not (t2.shape == adc.shape == dwi.shape):
        raise ValueError(
            f"Grid mismatch for {code}: "
            f"T2={t2.shape}, ADC={adc.shape}, DWI={dwi.shape}"
        )

    t2 = robust_zscore(t2)
    adc = robust_zscore(adc)
    dwi = robust_zscore(dwi)

    # [C, X, Y, Z]
    volume = np.stack([t2, adc, dwi], axis=0)
    return volume.astype(np.float32)


In [6]:
# CELL 6 — 3D patient dataset

class Patient3DDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        volume = load_patient_volume(row["code"])

        x = torch.from_numpy(volume)

        target = -1
        if pd.notna(row["target"]):
            target = int(row["target"])

        return {
            "code": str(row["code"]),
            "image": x,
            "target": target
        }

train_df = patients[patients["split"].str.lower() == "train"].copy()
val_df   = patients[patients["split"].str.lower().isin(["val", "valid", "validation"])].copy()
test_df  = patients[patients["split"].str.lower() == "test"].copy()

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


Train: 38
Val: 29
Test: 30


In [7]:
# CELL 7 — Safe 3D resizing utility, and 3D augmentation

def resize_3d(x, target_size):
    """
    x: [C, X, Y, Z] -> [C, target_X, target_Y, target_Z]

    Trilinear-resizes the whole volume instead of cropping, so the full
    field of view (and therefore the prostate, wherever it sits in-plane)
    is always retained -- important since none of the base models here
    localize the gland first.
    """
    assert x.ndim == 4
    x = x.unsqueeze(0)  # [1,C,X,Y,Z]
    x = F.interpolate(x, size=target_size, mode="trilinear", align_corners=False)
    return x.squeeze(0)

def prepare_3d(x):
    # x is [C, X, Y, Z]
    x = resize_3d(x, PATCH_SIZE)
    return x.unsqueeze(0)  # [1,C,X,Y,Z]

def random_crop_3d(x, min_frac=0.85):
    """Random zoom-ish crop, applied before resize_3d. x: [C,X,Y,Z]."""
    c, X, Y, Z = x.shape
    frac = random.uniform(min_frac, 1.0)
    cx, cy, cz = max(int(X * frac), 1), max(int(Y * frac), 1), max(int(Z * frac), 1)
    sx = random.randint(0, X - cx)
    sy = random.randint(0, Y - cy)
    sz = random.randint(0, Z - cz)
    return x[:, sx:sx + cx, sy:sy + cy, sz:sz + cz]

def augment_volume_3d(x):
    """
    Light, label-preserving augmentation for a [C,X,Y,Z] volume: random
    left-right / anterior-posterior flip, random zoom-crop, and a small
    random intensity scale/shift (the volume is already z-scored, so a
    small affine jitter stays in a realistic range). Used only when
    fitting probes -- never for val/test prediction.
    """
    if random.random() < 0.5:
        x = torch.flip(x, dims=[1])
    if random.random() < 0.5:
        x = torch.flip(x, dims=[2])
    x = random_crop_3d(x, min_frac=0.85)
    scale = 1.0 + random.uniform(-0.1, 0.1)
    shift = random.uniform(-0.1, 0.1)
    return x * scale + shift


In [8]:
# CELL 8 — MONAI Swin UNETR encoder (frozen, feature-extraction only)

from monai.networks.nets import SwinUNETR

def expand_first_conv_channels(conv, in_channels):
    """
    Returns a new Conv/Conv3d-like module with `in_channels` input channels,
    whose weights are the pretrained single-channel kernel replicated across
    channels (and rescaled), so pretrained features survive the channel
    change instead of being thrown away.
    """
    if conv.in_channels == in_channels:
        return conv
    w = conv.weight.data  # [out, 1, ...]
    new_w = w.repeat(1, in_channels, *([1] * (w.ndim - 2))) / in_channels
    new_conv = type(conv)(
        in_channels, conv.out_channels,
        kernel_size=conv.kernel_size, stride=conv.stride,
        padding=conv.padding, bias=(conv.bias is not None),
    )
    new_conv.weight.data = new_w
    if conv.bias is not None:
        new_conv.bias.data = conv.bias.data.clone()
    return new_conv


class SwinUNETRClassifier(nn.Module):
    """
    Frozen Swin UNETR encoder (swinViT), used purely as a feature
    extractor. `extract_features` returns the globally-pooled deepest-
    stage embedding. Classification happens downstream, via a linear
    probe (logistic regression) fit on these frozen features against
    real train_df labels -- see the "frozen-feature linear probes" cell.
    Nothing inside this module is trained; that's the point (38 labeled
    patients is nowhere near enough to fine-tune a transformer without
    overfitting, but it's plenty to fit a linear probe on top of good
    pretrained features).

    The encoder is initialized from MONAI's official self-supervised
    pretrained weights (Tang et al., CVPR 2022), trained with a single
    input channel. `load_pretrained_encoder` loads those weights at 1
    channel, then expands the patch-embedding conv to `in_channels` by
    replicating the pretrained kernel -- the same trick used for
    MedicalNet below.

    NOTE: SwinUNETR() also allocates a full segmentation decoder that we
    never call. That costs some extra GPU memory on a 4 GB card; if you
    hit OOM, lowering `feature_size` is the simplest fix.
    """
    def __init__(self, img_size=PATCH_SIZE, in_channels=3, feature_size=48):
        super().__init__()

        # Recent MONAI releases dropped the `img_size` constructor arg
        # (the model is now input-size-agnostic internally; the only
        # real constraint is that each spatial dim of the input you pass
        # to forward() be divisible by patch_size**5 = 32). Try the old
        # signature first, fall back to the new one.
        try:
            self.backbone = SwinUNETR(
                img_size=img_size,
                in_channels=1,      # matches the SSL checkpoint; expanded below
                out_channels=1,
                feature_size=feature_size,
                spatial_dims=3,
                use_checkpoint=False,
            )
        except TypeError:
            self.backbone = SwinUNETR(
                in_channels=1,
                out_channels=1,
                feature_size=feature_size,
                spatial_dims=3,
                use_checkpoint=False,
            )
        self.img_size = img_size
        self.in_channels = in_channels

    def load_pretrained_encoder(self, path):
        ckpt = torch.load(path, map_location="cpu")
        state = clean_state_dict(ckpt)
        # Keep only encoder weights; drop SSL-only heads (rotation/
        # contrastive/reconstruction) that have no counterpart here.
        encoder_prefixes = ("patch_embed.", "layers1.", "layers2.", "layers3.", "layers4.", "norm.")
        state = {k: v for k, v in state.items() if k.startswith(encoder_prefixes)}

        missing, unexpected = self.backbone.swinViT.load_state_dict(state, strict=False)
        print(f"\nLoaded Swin SSL encoder: {path}")
        print("Missing keys:", len(missing))
        print("Unexpected keys:", len(unexpected))

        self.backbone.swinViT.patch_embed.proj = expand_first_conv_channels(
            self.backbone.swinViT.patch_embed.proj, self.in_channels
        )

        # Freeze -- we only ever train the linear probe on top of this.
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def extract_features(self, x):
        feat = self.backbone.swinViT(x, normalize=True)[-1]
        return F.adaptive_avg_pool3d(feat, 1).flatten(1)

    forward = extract_features


### Swin checkpoint note

Swin UNETR is normally a **segmentation architecture**, so there is no universally valid operation called "load arbitrary Swin UNETR weights and classify."

This notebook uses MONAI's official **self-supervised pretrained encoder** (Tang et al., CVPR 2022 — trained on unlabeled CT volumes with a rotation / contrastive / inpainting objective, no classification labels at all). The encoder (`swinViT`) is loaded from that checkpoint and kept **frozen**; the segmentation decoder is discarded entirely.

Classification comes from a separate linear probe (logistic regression) fit on this encoder's frozen, pooled features against real `train_df` labels — see the "frozen-feature linear probes" cell. The encoder itself is never updated.


In [9]:
# CELL 9 — MedicalNet 3D ResNet-50 encoder (frozen, feature-extraction only)

import medicalnet_resnet as resnet  # vendored Tencent MedicalNet architecture (external_models/)

class MedicalNetResNet50Classifier(nn.Module):
    """
    Frozen MedicalNet 3D ResNet-50 encoder used purely as a feature
    extractor. The official checkpoint is a *segmentation* network
    (1-channel input, conv_seg decoder); we keep only the encoder
    (conv1..layer4), drop conv_seg, and globally pool its output.
    conv1 is expanded from 1 to `in_channels` channels by replicating
    the pretrained kernel (see expand_first_conv_channels in CELL 8).

    Classification happens downstream, via a linear probe fit on these
    frozen features against real train_df labels -- nothing in this
    module is trained.
    """
    def __init__(self, in_channels=3):
        super().__init__()

        self.backbone = resnet.resnet50(
            sample_input_W=PATCH_SIZE[0],
            sample_input_H=PATCH_SIZE[1],
            sample_input_D=PATCH_SIZE[2],
            shortcut_type="B",
            no_cuda=not torch.cuda.is_available(),
            num_seg_classes=2,
        )
        del self.backbone.conv_seg
        self.in_channels = in_channels

    def load_pretrained(self, path):
        ckpt = torch.load(path, map_location="cpu")
        state = clean_state_dict(ckpt)
        state = {k: v for k, v in state.items() if not k.startswith("conv_seg")}

        missing, unexpected = self.backbone.load_state_dict(state, strict=False)
        print(f"\nLoaded MedicalNet encoder: {path}")
        print("Missing keys:", len(missing))
        print("Unexpected keys:", len(unexpected))

        self.backbone.conv1 = expand_first_conv_channels(self.backbone.conv1, self.in_channels)

        # Freeze -- we only ever train the linear probe on top of this.
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def extract_features(self, x):
        b = self.backbone
        x = b.conv1(x)
        x = b.bn1(x)
        x = b.relu(x)
        x = b.maxpool(x)
        x = b.layer1(x)
        x = b.layer2(x)
        x = b.layer3(x)
        x = b.layer4(x)
        return F.adaptive_avg_pool3d(x, 1).flatten(1)

    forward = extract_features


In [10]:
# CELL 10 — 2D model wrappers (ImageNet-pretrained, frozen, feature-extraction only)
#
# True RadImageNet PyTorch weights for InceptionResNetV2 aren't publicly
# available from a safe, state-dict-only source (only a Keras/TF release,
# plus a few HF repos that ship full pickled nn.Module objects, which we
# avoid loading here). Both 2D models therefore use standard ImageNet
# pretrained weights -- still genuine pretrained features, just not
# radiology-domain pretrained.
#
# Both wrappers drop their original ImageNet classifier entirely and
# expose only pooled features; classification happens downstream via a
# linear probe fit on frozen features (see the "frozen-feature linear
# probes" cell).

import torchvision
from torchvision.models import densenet121, DenseNet121_Weights

class DenseNet121Classifier(nn.Module):
    INPUT_SIZE = 224

    def __init__(self):
        super().__init__()
        self.model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def extract_features(self, x):
        # Same pooling densenet121 uses internally, just stopping before
        # its (unused, ImageNet-1000-way) classifier.
        f = self.model.features(x)
        f = F.relu(f, inplace=False)
        f = F.adaptive_avg_pool2d(f, 1).flatten(1)
        return f

    forward = extract_features


class InceptionResNetV2Classifier(nn.Module):
    INPUT_SIZE = 299

    def __init__(self):
        super().__init__()
        import timm

        # num_classes=0 drops the classifier head entirely -- forward()
        # then returns pooled features directly.
        self.model = timm.create_model(
            "inception_resnet_v2",
            pretrained=True,
            num_classes=0,
        )
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def extract_features(self, x):
        return self.model(x)

    forward = extract_features

print("2D feature extractors ready (ImageNet-pretrained, frozen backbones).")


2D feature extractors ready (ImageNet-pretrained, frozen backbones).


In [11]:
# CELL 11 — Checkpoint state-dict cleanup

def clean_state_dict(state):
    """Unwraps common checkpoint containers and strips a DataParallel
    'module.' prefix, without touching the actual tensor values."""
    if isinstance(state, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in state and isinstance(state[key], dict):
                state = state[key]

    cleaned = {}
    for k, v in state.items():
        if k.startswith("module."):
            k = k[7:]
        cleaned[k] = v
    return cleaned


In [12]:
# CELL 12 — 2.5D conversion for the 2D models, and 2D augmentation

def make_2p5d_batch(volume, z_index, radius=1):
    """
    volume: torch tensor [C, X, Y, Z]
    Returns [3, H, W].

    We use neighboring axial slices from the T2 modality for RGB-like
    channel construction. This avoids pretending T2/ADC/DWI are RGB.

    If your RadImageNet fine-tuning used T2/ADC/DWI as the three channels,
    replace this function with that exact training preprocessing.
    """
    z = volume.shape[-1]

    indices = [
        max(0, min(z - 1, z_index + dz))
        for dz in range(-radius, radius + 1)
    ]

    # Use T2 neighboring slices as 2.5D channels.
    slices = [volume[0, :, :, i] for i in indices]

    x = torch.stack(slices, dim=0)
    return x

def augment_slice_2d(x):
    """
    Light, label-preserving augmentation for a [3,H,W] 2.5D slice stack:
    random horizontal flip, random zoom-crop, and a small random intensity
    scale/shift. Used only when fitting probes -- never for val/test
    prediction.
    """
    if random.random() < 0.5:
        x = torch.flip(x, dims=[2])
    c, H, W = x.shape
    frac = random.uniform(0.85, 1.0)
    ch, cw = max(int(H * frac), 1), max(int(W * frac), 1)
    sh = random.randint(0, H - ch)
    sw = random.randint(0, W - cw)
    x = x[:, sh:sh + ch, sw:sw + cw]
    scale = 1.0 + random.uniform(-0.1, 0.1)
    shift = random.uniform(-0.1, 0.1)
    return x * scale + shift

def resize_2d(x, size):
    return F.interpolate(
        x.unsqueeze(0),
        size=(size, size),
        mode="bilinear",
        align_corners=False,
    ).squeeze(0)

# ImageNet-style normalization used by standard 2D CNN input wrappers.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def prepare_2d(x, size=224):
    x = resize_2d(x, size)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x


## Stacking strategy

Each base model $M_i$ here is itself a frozen pretrained backbone + a linear probe fit on (augmented) frozen features (see CELL 22) — so $p_i$ below is a real, trained prediction, not a random one.

We first obtain **out-of-fold predictions** from each base model, across all labeled (train+val) patients via K-fold cross-validation:

\[
p_i =
[P(y=1|x,M_i)]
\]

and construct:

\[
Z =
[p_{Swin},p_{MedNet},p_{DenseNet},p_{Inception}]
\]

Then train a meta-classifier:

\[
\hat y =
g(Z)
\]

using logistic regression.

This is genuine stacking; it is preferable to averaging because the meta-model can learn that some base models are more reliable than others.


In [13]:
# CELL 13 — Frozen-feature extraction, with on-disk caching and optional
# augmentation (augmented copies are for probe *training* only -- always
# call with n_aug=0 when generating val/test predictions).

FEATURES_DIR = OUTPUT_DIR / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

@torch.no_grad()
def extract_3d_features(model, df, cache_name, n_aug=0):
    """
    One feature vector per patient (whole-volume models: MedicalNet, Swin).
    n_aug > 0 additionally extracts n_aug randomly-augmented copies per
    patient (augment_volume_3d, CELL 7), giving the downstream linear probe
    more training examples -- the frozen backbone needs no augmented
    gradients, just augmented inputs.
    """
    cache_path = FEATURES_DIR / f"{cache_name}.npz"
    if cache_path.exists():
        d = np.load(cache_path, allow_pickle=True)
        return d["codes"], d["features"], d["targets"]

    codes, feats, targets = [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=cache_name):
        code = str(row["code"])
        target = row["target"] if pd.notna(row["target"]) else np.nan
        volume = torch.from_numpy(load_patient_volume(code))

        variants = [volume] + [augment_volume_3d(volume) for _ in range(n_aug)]
        for v in variants:
            x = prepare_3d(v).to(DEVICE)
            f = model.extract_features(x)[0].cpu().numpy()
            codes.append(code)
            feats.append(f)
            targets.append(target)

    codes = np.array(codes)
    feats = np.stack(feats)
    targets = np.array(targets, dtype=np.float32)
    np.savez(cache_path, codes=codes, features=feats, targets=targets)
    return codes, feats, targets

@torch.no_grad()
def extract_2d_features(model, df, cache_name, input_size, n_aug=0):
    """
    One feature vector per (patient, slice) for the 2.5D models
    (DenseNet-121, InceptionResNetV2). Every slice inherits its patient's
    label -- a weak-supervision approximation, since we only have
    patient-level lesion labels here, not per-slice ones. n_aug > 0
    additionally extracts n_aug randomly-augmented copies per slice
    (augment_slice_2d, CELL 12).
    """
    cache_path = FEATURES_DIR / f"{cache_name}.npz"
    if cache_path.exists():
        d = np.load(cache_path, allow_pickle=True)
        return d["codes"], d["slice_idx"], d["features"], d["targets"]

    codes, slice_idx, feats, targets = [], [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=cache_name):
        code = str(row["code"])
        target = row["target"] if pd.notna(row["target"]) else np.nan
        volume = torch.from_numpy(load_patient_volume(code))
        z = volume.shape[-1]
        for zi in range(z):
            base_slice = make_2p5d_batch(volume, zi, NEIGHBOR_RADIUS)
            variants = [base_slice] + [augment_slice_2d(base_slice) for _ in range(n_aug)]
            for v in variants:
                x = prepare_2d(v, input_size).unsqueeze(0).to(DEVICE)
                f = model.extract_features(x)[0].cpu().numpy()
                codes.append(code)
                slice_idx.append(zi)
                feats.append(f)
                targets.append(target)

    codes = np.array(codes)
    slice_idx = np.array(slice_idx)
    feats = np.stack(feats)
    targets = np.array(targets, dtype=np.float32)
    np.savez(cache_path, codes=codes, slice_idx=slice_idx, features=feats, targets=targets)
    return codes, slice_idx, feats, targets

print("Feature-extraction helpers ready.")


Feature-extraction helpers ready.


In [14]:
# CELL 14 — Load the three directly usable encoder models (frozen)

ensure_checkpoint(MEDICALNET_CKPT, MEDICALNET_URL)
medicalnet_model = MedicalNetResNet50Classifier(in_channels=3)
medicalnet_model.load_pretrained(MEDICALNET_CKPT)
medicalnet_model = medicalnet_model.to(DEVICE).eval()

rad_dense_model = DenseNet121Classifier().to(DEVICE).eval()
rad_inception_model = InceptionResNetV2Classifier().to(DEVICE).eval()

print("Three frozen feature extractors loaded.")



Loaded MedicalNet encoder: C:\Users\adity\Downloads\aiims_public\checkpoints\medicalnet_resnet50_23datasets.pth
Missing keys: 0
Unexpected keys: 0
Three frozen feature extractors loaded.


In [15]:
# CELL 15 — Load Swin UNETR encoder (self-supervised pretrained, frozen)

ensure_checkpoint(SWIN_CKPT, SWIN_SSL_URL)
swin_model = SwinUNETRClassifier(img_size=PATCH_SIZE, in_channels=3, feature_size=48)
swin_model.load_pretrained_encoder(SWIN_CKPT)
swin_model = swin_model.to(DEVICE).eval()

print("Swin UNETR encoder loaded.")



Loaded Swin SSL encoder: C:\Users\adity\Downloads\aiims_public\checkpoints\model_swinvit.pt
Missing keys: 32
Unexpected keys: 34
Swin UNETR encoder loaded.


In [16]:
# CELL 16 — Base-model prediction functions (frozen features -> linear probe)
#
# `probes` is a dict of fitted sklearn LogisticRegression models, one per
# base model, created in the "frozen-feature linear probes" execution cell
# below. These functions only read it, so they must be called after that
# cell has run.

@torch.no_grad()
def predict_medicalnet(volume):
    x = prepare_3d(volume).to(DEVICE)
    f = medicalnet_model.extract_features(x).cpu().numpy()
    return float(probes["medicalnet"].predict_proba(f)[0, 1])

@torch.no_grad()
def predict_rad_models(volume):
    """
    Slice-wise 2.5D prediction.
    Returns the mean probability over valid axial slices.
    """
    z = volume.shape[-1]
    probs_dense = []
    probs_inc = []

    for zi in range(z):
        x = make_2p5d_batch(volume, zi, NEIGHBOR_RADIUS)

        x_dense = prepare_2d(x, DenseNet121Classifier.INPUT_SIZE).unsqueeze(0).to(DEVICE)
        x_inc = prepare_2d(x, InceptionResNetV2Classifier.INPUT_SIZE).unsqueeze(0).to(DEVICE)

        f_dense = rad_dense_model.extract_features(x_dense).cpu().numpy()
        f_inc = rad_inception_model.extract_features(x_inc).cpu().numpy()

        probs_dense.append(float(probes["rad_densenet"].predict_proba(f_dense)[0, 1]))
        probs_inc.append(float(probes["rad_inception"].predict_proba(f_inc)[0, 1]))

    return float(np.mean(probs_dense)), float(np.mean(probs_inc))

@torch.no_grad()
def predict_swin(volume):
    x = prepare_3d(volume).to(DEVICE)
    f = swin_model.extract_features(x).cpu().numpy()
    return float(probes["swin"].predict_proba(f)[0, 1])


In [17]:
# CELL 17 — Generate validation base predictions

def generate_base_predictions(df, include_swin=True):
    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        code = str(row["code"])
        volume = torch.from_numpy(load_patient_volume(code))

        med_prob = predict_medicalnet(volume)
        dense_prob, inc_prob = predict_rad_models(volume)

        if include_swin:
            swin_prob = predict_swin(volume)
        else:
            swin_prob = np.nan

        records.append({
            "code": code,
            "target": int(row["target"]),
            "swin": swin_prob,
            "medicalnet": med_prob,
            "rad_densenet": dense_prob,
            "rad_inception": inc_prob
        })

    return pd.DataFrame(records)

print("Function ready.")


Function ready.


In [18]:
# CELL 18 — Train stacking meta-learner

def train_stacker(val_base):
    feature_cols = [
        "swin",
        "medicalnet",
        "rad_densenet",
        "rad_inception"
    ]

    if val_base[feature_cols].isna().any().any():
        raise ValueError(
            "Some base predictions are missing. "
            "All four base models must produce validation predictions."
        )

    X = val_base[feature_cols].values
    y = val_base["target"].values

    meta = LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced"
    )

    meta.fit(X, y)

    print("Stacker coefficients:")
    print(pd.Series(meta.coef_[0], index=feature_cols))
    print("Intercept:", meta.intercept_[0])

    return meta

# Example (see CELL 22, which actually calls this on out-of-fold predictions):
# oof_base = pd.read_csv(OUTPUT_DIR / "oof_base_predictions.csv")
# stacker = train_stacker(oof_base)


In [19]:
# CELL 19 — Evaluate stacking

def evaluate_stacker(meta, val_base):
    feature_cols = [
        "swin",
        "medicalnet",
        "rad_densenet",
        "rad_inception"
    ]

    X = val_base[feature_cols].values
    y = val_base["target"].values

    p = meta.predict_proba(X)[:,1]
    pred = (p >= 0.5).astype(int)

    print("Accuracy:", accuracy_score(y, pred))
    print("Balanced accuracy:", balanced_accuracy_score(y, pred))
    print("Macro F1:", f1_score(y, pred, average="macro"))

    if len(np.unique(y)) == 2:
        print("ROC-AUC:", roc_auc_score(y, p))

    print("\nClassification report:")
    print(classification_report(y, pred, digits=4))

    print("\nConfusion matrix:")
    print(confusion_matrix(y, pred))

    return p, pred


In [20]:
# CELL 20 — Slice-level inference (frozen features -> linear probe)

@torch.no_grad()
def predict_slice_scores(code):
    """
    Produces one score for every axial slice, as required by the dataset README.
    Slice score = average of the two 2D models' probe-predicted probabilities.
    """
    volume = torch.from_numpy(load_patient_volume(code))

    z = volume.shape[-1]
    records = []

    for zi in range(z):
        x = make_2p5d_batch(volume, zi, NEIGHBOR_RADIUS)

        x_dense = prepare_2d(x, DenseNet121Classifier.INPUT_SIZE).unsqueeze(0).to(DEVICE)
        x_inc = prepare_2d(x, InceptionResNetV2Classifier.INPUT_SIZE).unsqueeze(0).to(DEVICE)

        f_dense = rad_dense_model.extract_features(x_dense).cpu().numpy()
        f_inc = rad_inception_model.extract_features(x_inc).cpu().numpy()

        p_dense = float(probes["rad_densenet"].predict_proba(f_dense)[0, 1])
        p_inc = float(probes["rad_inception"].predict_proba(f_inc)[0, 1])

        score = 0.5 * (p_dense + p_inc)

        records.append({
            "code": code,
            "slice_index": zi,
            "score": score
        })

    return pd.DataFrame(records)


In [21]:
# CELL 21 — Patient-level TEST inference

def run_test_patient_inference(test_df, meta):
    feature_cols = [
        "swin",
        "medicalnet",
        "rad_densenet",
        "rad_inception"
    ]

    patient_rows = []
    slice_frames = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        code = str(row["code"])
        volume = torch.from_numpy(load_patient_volume(code))

        # ---- Base model predictions ----
        p_swin = predict_swin(volume)
        p_med = predict_medicalnet(volume)
        p_dense, p_inc = predict_rad_models(volume)

        X = np.array([[p_swin, p_med, p_dense, p_inc]], dtype=np.float32)
        final_score = float(meta.predict_proba(X)[0,1])

        patient_rows.append({
            "code": code,
            "score": final_score
        })

        # ---- Required slice scores ----
        slice_frames.append(
            predict_slice_scores(code)
        )

    patient_scores = pd.DataFrame(patient_rows)
    slice_scores = pd.concat(slice_frames, ignore_index=True)

    patient_path = OUTPUT_DIR / "patient_scores.csv"
    slice_path = OUTPUT_DIR / "slice_scores.csv"

    patient_scores.to_csv(patient_path, index=False)
    slice_scores.to_csv(slice_path, index=False)

    print("Saved:")
    print(patient_path)
    print(slice_path)

    return patient_scores, slice_scores


## Recommended execution order

Run the notebook in this order:

1. Run CELL 1–7 (imports, config, CSV checks, data loading, resize + 3D augmentation utilities).
2. Run CELL 8–13 (model definitions, 2D augmentation, feature-extraction helpers). CELL 14–15 will auto-download the MedicalNet and Swin UNETR SSL pretrained checkpoints into `checkpoints/` the first time they run (a few hundred MB, one-time).
3. Run CELL 14 (MedicalNet / DenseNet-121 / InceptionResNetV2) and CELL 15 (Swin UNETR) — all four backbones load frozen.
4. Run CELL 16–19 (prediction-function definitions; `train_stacker`/`evaluate_stacker`).
5. Run CELL 22: this is the one that actually trains something, and it's the expensive one the first time (uncached). It combines `train_df` + `val_df` into `labeled_df` (67 patients), runs 5-fold cross-validation — each fold fitting augmented probes on the other folds and predicting the held-out one — to get leakage-free out-of-fold predictions for every labeled patient, fits/evaluates the stacker on those, then refits "production" probes on all 67 patients for the test run. Features are cached under `ensemble_outputs/features/`; delete that folder to force recomputation (e.g. after changing `N_AUG`).
6. Run CELL 23: test-set inference, using the production probes + stacker from CELL 22.
7. Submit from `ensemble_outputs/`:
   - `patient_scores.csv`
   - `slice_scores.csv`

### What "fine-tuned" means here

Every backbone stays frozen; only four small linear probes (one per base model, refit per CV fold and once more on all labeled data) plus the stacker are actually trained, on this dataset's real labels — augmented with `N_AUG` extra copies per patient/slice (see the intro cell). That's still nowhere near enough data to fine-tune 3D CNNs/transformers end-to-end without overfitting, which is why the backbones stay frozen.

### Leakage rule — now handled by CELL 22 itself

Don't fit the stacker on predictions from a probe that was trained on those same patients. CELL 22 already avoids this via K-fold cross-validation: each fold's held-out predictions come from probes that never saw those patients, so the full 67-patient OOF set the stacker trains on is leakage-free, even though every patient also contributed to *some* probe's training data.

```text
labeled_df (train+val, 67 patients)
  ├── fold 1 → fit probes (augmented) on folds 2-5 → predict fold 1
  ├── fold 2 → fit probes (augmented) on folds 1,3-5 → predict fold 2
  ├── ...
  └── fold 5 → fit probes (augmented) on folds 1-4 → predict fold 5

OOF predictions (all 67 patients, each from a probe that never saw it)
      ↓
  train + evaluate stacker
      ↓
refit "production" probes on all 67 labeled patients
      ↓
   held-out test (CELL 23)
```


In [22]:
# CELL 22 — K-fold out-of-fold stacking (train+val) with augmented linear probes
#
# train_df alone is small (38 patients). ALL labeled patients (train_df +
# val_df, 67 total) are now used to fit each base model's linear probe,
# and probes are trained on augmented copies of each patient/slice (flips
# + small zoom-crop + intensity jitter -- augment_volume_3d / augment_slice_2d
# in CELL 7 / CELL 12), not just the original volume/slices.
#
# To use every labeled patient for probe training WITHOUT letting any
# probe's validation prediction leak information about a patient it was
# trained on, we use K-fold cross-validation: each fold fits fresh probes
# on the other folds only, then predicts the held-out fold. Those
# out-of-fold (OOF) predictions -- one per labeled patient, always from a
# probe that never saw that patient during training -- are what the
# stacker is fit and evaluated on. Augmentation is applied only when
# FITTING probes, never when generating OOF/test predictions, so the
# reported metrics stay honest.
#
# After cross-validation, the "production" probes actually used for test
# inference below are refit one more time on ALL labeled patients (still
# augmented), to make use of every labeled example at inference time.

from sklearn.model_selection import StratifiedKFold

labeled_df = pd.concat([train_df, val_df], ignore_index=True).reset_index(drop=True)
print("Labeled patients (train+val) used for probe fitting:", len(labeled_df))

def fit_base_probe(name, df, cache_name, n_aug=0):
    if name == "medicalnet":
        _, feats, y = extract_3d_features(medicalnet_model, df, cache_name, n_aug=n_aug)
    elif name == "swin":
        _, feats, y = extract_3d_features(swin_model, df, cache_name, n_aug=n_aug)
    elif name == "rad_densenet":
        _, _, feats, y = extract_2d_features(rad_dense_model, df, cache_name, DenseNet121Classifier.INPUT_SIZE, n_aug=n_aug)
    elif name == "rad_inception":
        _, _, feats, y = extract_2d_features(rad_inception_model, df, cache_name, InceptionResNetV2Classifier.INPUT_SIZE, n_aug=n_aug)
    else:
        raise ValueError(name)
    return LogisticRegression(max_iter=2000, class_weight="balanced").fit(feats, y.astype(int))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_frames = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(labeled_df["code"], labeled_df["target"])):
    fold_train = labeled_df.iloc[tr_idx].reset_index(drop=True)
    fold_val = labeled_df.iloc[va_idx].reset_index(drop=True)
    print(f"\n=== Fold {fold + 1}/{N_FOLDS}: train={len(fold_train)} val={len(fold_val)} ===")

    # Fit this fold's probes (augmented) and point the global `probes` dict
    # -- read by predict_medicalnet/predict_swin/predict_rad_models -- at
    # them, so generate_base_predictions can be reused unchanged below.
    probes = {
        "medicalnet": fit_base_probe("medicalnet", fold_train, f"fold{fold}_medicalnet_train", n_aug=N_AUG),
        "swin": fit_base_probe("swin", fold_train, f"fold{fold}_swin_train", n_aug=N_AUG),
        "rad_densenet": fit_base_probe("rad_densenet", fold_train, f"fold{fold}_densenet_train", n_aug=N_AUG),
        "rad_inception": fit_base_probe("rad_inception", fold_train, f"fold{fold}_inception_train", n_aug=N_AUG),
    }

    oof_frames.append(generate_base_predictions(fold_val, include_swin=True))

oof_base = pd.concat(oof_frames, ignore_index=True)
oof_base.to_csv(OUTPUT_DIR / "oof_base_predictions.csv", index=False)

print(f"\n=== Stacker on out-of-fold predictions (all {len(oof_base)} labeled patients) ===")
stacker = train_stacker(oof_base)
_ = evaluate_stacker(stacker, oof_base)

# --- Production probes: refit on ALL labeled patients (train+val), used
#     by the test-set inference cell below. ---
probes = {
    "medicalnet": fit_base_probe("medicalnet", labeled_df, "final_medicalnet_train", n_aug=N_AUG),
    "swin": fit_base_probe("swin", labeled_df, "final_swin_train", n_aug=N_AUG),
    "rad_densenet": fit_base_probe("rad_densenet", labeled_df, "final_densenet_train", n_aug=N_AUG),
    "rad_inception": fit_base_probe("rad_inception", labeled_df, "final_inception_train", n_aug=N_AUG),
}

import joblib
joblib.dump(probes, OUTPUT_DIR / "frozen_feature_probes.pkl")
joblib.dump(stacker, OUTPUT_DIR / "stacker.pkl")
print("\nFinal probes + stacker saved.")


Labeled patients (train+val) used for probe fitting: 67

=== Fold 1/5: train=53 val=14 ===


fold0_medicalnet_train:   0%|          | 0/53 [00:00<?, ?it/s]

fold0_swin_train:   0%|          | 0/53 [00:00<?, ?it/s]

fold0_densenet_train:   0%|          | 0/53 [00:00<?, ?it/s]

fold0_inception_train:   0%|          | 0/53 [00:00<?, ?it/s]

...

final_medicalnet_train:   0%|          | 0/67 [00:00<?, ?it/s]

final_swin_train:   0%|          | 0/67 [00:00<?, ?it/s]

final_densenet_train:   0%|          | 0/67 [00:00<?, ?it/s]

final_inception_train:   0%|          | 0/67 [00:00<?, ?it/s]


Final probes + stacker saved.


In [23]:
# CELL 23 — Final test execution

patient_scores, slice_scores = run_test_patient_inference(
    test_df,
    stacker
)

display(patient_scores.head())
display(slice_scores.head())


  0%|          | 0/30 [00:00<?, ?it/s]

Saved:
C:\Users\adity\Downloads\aiims_public\ensemble_outputs\patient_scores.csv
C:\Users\adity\Downloads\aiims_public\ensemble_outputs\slice_scores.csv


,code,score
0,C0004,0.430942
1,C0005,0.512580
2,C0006,0.529925
3,C0007,0.467620
4,C0014,0.399572


,code,slice_index,score
0,C0004,0,0.582971
1,C0004,1,0.709262
2,C0004,2,0.797151
3,C0004,3,0.599060
4,C0004,4,0.379740


In [24]:
# CELL 24 — Optional: save complete ensemble configuration

config = {
    "target": "binary: 0=PI-RADS 1-2, 1=PI-RADS 3-5",
    "modalities": ["t2", "adc", "dwi_highb"],
    "models": [
        "Swin UNETR (MONAI, SSL-pretrained frozen encoder + linear probe)",
        "MedicalNet 3D ResNet-50 (pretrained frozen encoder + linear probe)",
        "DenseNet-121 (ImageNet-pretrained frozen encoder + linear probe)",
        "InceptionResNetV2 (ImageNet-pretrained frozen encoder + linear probe)"
    ],
    "fine_tuned_on_this_dataset": True,
    "fine_tuning_method": "linear probing on frozen pretrained features (logistic regression)",
    "probe_training_data": "train_df + val_df combined, via K-fold cross-validation",
    "n_folds": N_FOLDS,
    "n_augmented_copies_per_sample": N_AUG,
    "stacker": "LogisticRegression(class_weight='balanced'), fit on out-of-fold base predictions",
    "patch_size": PATCH_SIZE,
    "neighbor_radius": NEIGHBOR_RADIUS,
    "seed": SEED
}

with open(OUTPUT_DIR / "ensemble_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


{
  "target": "binary: 0=PI-RADS 1-2, 1=PI-RADS 3-5",
  "modalities": [
    "t2",
    "adc",
    "dwi_highb"
  ],
  "models": [
    "Swin UNETR (MONAI, SSL-pretrained frozen encoder + linear probe)",
    "MedicalNet 3D ResNet-50 (pretrained frozen encoder + linear probe)",
    "DenseNet-121 (ImageNet-pretrained frozen encoder + linear probe)",
    "InceptionResNetV2 (ImageNet-pretrained frozen encoder + linear probe)"
  ],
  "fine_tuned_on_this_dataset": true,
  "fine_tuning_method": "linear probing on frozen pretrained features (logistic regression)",
  "probe_training_data": "train_df + val_df combined, via K-fold cross-validation",
  "n_folds": 5,
  "n_augmented_copies_per_sample": 4,
  "stacker": "LogisticRegression(class_weight='balanced'), fit on out-of-fold base predictions",
  "patch_size": [
    96,
    96,
    64
  ],
  "neighbor_radius": 1,
  "seed": 42
}
